In [5]:
# Imports, settings, and file paths.
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

MIN_ANSWERED = 3             # drop respondents with fewer core attributes than this
TOP_COUNTRIES = 25           # the rest become "Other"
TOP_TECH = {"lang": 30, "db": 15, "platform": 15, "web": 15}
IQR_K = 1.5                  # Tukey fence multiplier for log10(compensation)
FIRST_EXACT_COMP_YEAR = 2017 # earlier years report salary bands, which are not fenced
AGE_RANGE = (10, 90)
MAX_YEARS = 70
PCA_VARIANCE = 0.90
DECIMALS = 3

OUTPUT_DIR = Path.cwd()
INPUT_CSV = OUTPUT_DIR / "combined_survey_all_years.csv"
report = {}
start = time.time()

In [6]:
# Tables that map each survey year's columns and answers onto shared attributes.
ID_COLUMNS = ["Respondent", "ResponseId"]
NUMERIC_ATTRS = ["age", "years_code", "years_code_pro", "org_size_log10", "education_level", "job_satisfaction", "comp_usd"]
CATEGORICAL_ATTRS = ["country", "employment", "gender", "remote_work"]

# Column names per attribute. Each year uses at most one of them.
SOURCES = {
    "country": ["What Country or Region do you live in?", "What Country do you live in?", "_2015_Country", "country", "Country"],
    "age": ["How old are you?", "_2015_Age", "age_range", "Age"],
    "gender": ["What is your gender?", "_2015_Gender", "gender", "Gender"],
    "employment": ["_2015_Employment Status", "employment_status", "EmploymentStatus", "Employment"],
    "education": ["_2015_education", "education", "FormalEducation", "EdLevel"],
    "org_size": ["Which best describes the size of your company?", "How many people work for your company?",
                 "company_size_range", "CompanySize", "OrgSize"],
    "years_code": ["How many years of IT/Programming experience do you have?", "_2015_Years IT / Programming Experience",
                   "experience_range", "YearsProgram", "YearsCoding", "YearsCode"],
    "years_code_pro": ["YearsCodedJob", "YearsCodingProf", "YearsCodePro"],
    "remote_work": ["Do you work remotely?", "_2015_Remote Status", "remote", "HomeRemote", "WorkRemote", "RemoteWork"],
    "job_satisfaction": ["Please rate your job/career satisfaction", "What best describes your career / job satisfaction? ",
                         "What best describes your career / job satisfaction?", "_2015_Job Satisfaction",
                         "job_satisfaction", "JobSatisfaction", "JobSat"],
    "comp_band": ["Including bonus, what is your annual compensation in USD?", "_2015_Compensation: midpoint", "salary_midpoint"],
    "comp_exact": ["Salary_2017", "ConvertedSalary", "ConvertedComp", "ConvertedCompYearly"],
}
ROLE_SOURCES = ["Which of the following best describes your occupation?", "_2015_Occupation", "occupation",
                "DeveloperType", "WebDeveloperType", "DevType"]
TECH_SOURCES = {
    "lang": ["HaveWorkedLanguage", "LanguageWorkedWith", "LanguageHaveWorkedWith"],
    "db": ["HaveWorkedDatabase", "DatabaseWorkedWith", "DatabaseHaveWorkedWith"],
    "platform": ["HaveWorkedPlatform", "PlatformWorkedWith", "PlatformHaveWorkedWith"],
    "web": ["HaveWorkedFramework", "FrameworkWorkedWith", "WebFrameWorkedWith", "WebframeWorkedWith", "WebframeHaveWorkedWith"],
}
# 2011-2016 ask one mixed question; its answers are routed to families by name.
EARLY_LANGUAGE_QUESTIONS = {
    2011: ("Which languages are you proficient in?", range(31, 42)),
    2012: ("Which languages are you proficient in?", range(23, 36)),
    2013: ("Which of the following languages or technologies", range(57, 69)),
    2014: ("Which of the following languages or technologies", range(43, 53)),
}

# Excel turned these ranges into dates: 2/5/2013 is "2-5", 41435 is the serial for 6/10/2013.
EXCEL_DATE_YEARS = {
    "41310": 3.5, "41435": 8.0, "40944": 3.5, "41070": 8.0,
    "2/5/2013": 3.5, "6/10/2013": 8.0, "2/5/2014": 3.5, "6/10/2014": 8.0,
}
AGE_BANDS = {
    "< 20": 17, "<20": 17, "under 18 years old": 16, "20-24": 22, "25-29": 27, "30-34": 32, "35-39": 37,
    "40-49": 44.5, "40-50": 45, "50-59": 54.5, "51-60": 55.5, "> 60": 65, ">60": 65, "60+": 65,
    "18-24 years old": 21, "25-34 years old": 29.5, "35-44 years old": 39.5, "45-54 years old": 49.5,
    "55-64 years old": 59.5, "65 years or older": 70,
}
ORG_SIZES = {
    "start up (1-25)": 13, "mature small business (25-100)": 62, "mid sized (100-999)": 550, "fortune 1000 (1,000+)": 5000,
    "1/25/2013": 13, "26-100": 63, "101-999": 550, "1,000-3,000": 2000, "3001": 5000,
    "just me - i am a freelancer, sole proprietor, etc.": 1, "1-4 employees": 2.5, "5-9 employees": 7,
    "2-9 employees": 5.5, "2 to 9 employees": 5.5, "fewer than 10 employees": 5, "less than 20 employees": 10,
    "10-19 employees": 14.5, "10 to 19 employees": 14.5, "20-99 employees": 59.5, "20 to 99 employees": 59.5,
    "100-499 employees": 299.5, "100 to 499 employees": 299.5, "500-999 employees": 749.5, "500 to 999 employees": 749.5,
    "1,000-4,999 employees": 2999.5, "1,000 to 4,999 employees": 2999.5, "5,000-9,999 employees": 7499.5,
    "5,000 to 9,999 employees": 7499.5, "10,000+ employees": 15000, "10,000 or more employees": 15000,
}
COMP_BANDS = {
    "<$20,000": 10000, "$20,000 - $40,000": 30000, "$40,000 - $60,000": 50000, "$60,000 - $80,000": 70000,
    "$80,000 - $100,000": 90000, "$100,000 - $120,000": 110000, "$120,000 - $140,000": 130000, ">$140,000": 150000,
}
SATISFACTION_TEXT = {
    "fml": 0, "i hate my job": 0, "hate my job": 0, "i wish i had a job!": None,
    "very dissatisfied": 0, "extremely dissatisfied": 0, "moderately dissatisfied": 1 / 6, "slightly dissatisfied": 1 / 3,
    "i'm somewhat dissatisfied with my job": 0.25, "neither satisfied nor dissatisfied": 0.5,
    "i'm neither satisfied nor dissatisfied": 0.5, "i'm neither satisfied nor dissatisfied with my job": 0.5,
    "it pays the bills": 0.5, "its a paycheck": 0.5, "it's a paycheck": 0.5,
    "slightly satisfied": 2 / 3, "moderately satisfied": 5 / 6, "i'm somewhat satisfied with my job": 0.75,
    "i enjoy going to work": 0.75, "very satisfied": 1, "extremely satisfied": 1, "i love my job": 1,
    "love my job": 1, "so happy it hurts": 1,
}
REMOTE = {
    "never": "in_person", "occasionally": "in_person", "i rarely work remote": "in_person", "i rarely work remotely": "in_person",
    "a few days each month": "in_person", "less than once per month / never": "in_person", "full in-person": "in_person",
    "in-person": "in_person", "part-time remote": "hybrid", "less than half the time, but at least one day each week": "hybrid",
    "about half the time": "hybrid", "more than half, but not all, the time": "hybrid",
    "hybrid (some remote, some in-person)": "hybrid", "hybrid (some remote, leans heavy to in-person)": "hybrid",
    "hybrid (some in-person, leans heavy to flexibility)": "hybrid",
    "your choice (very flexible, you can come in when you want or just as needed)": "hybrid",
    "full-time remote": "remote", "all or almost all the time (i'm full-time remote)": "remote",
    "fully remote": "remote", "remote": "remote",
}
COUNTRY_ALIASES = {
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Russia": "Russian Federation",
    "Iran, Islamic Republic of...": "Iran",
    "Ireland {Republic}": "Ireland",
    "Vietnam": "Viet Nam",
    "Republic of Korea": "South Korea",
}
# Regional buckets from the 2011-2013 surveys cannot be matched to one country.
COUNTRY_BUCKETS = re.compile(r"^(other\b|south america|africa|middle east|central america|north america)", re.I)
ROLE_RULES = [
    ("full_stack", r"full.?stack"),
    ("back_end", r"back.?end|server programmer"),
    ("front_end", r"front.?end"),
    ("mobile", r"mobile|android|\bios\b|windows phone"),
    ("embedded", r"embedded|hardware"),
    ("game_graphics", r"game|graphics program"),
    ("desktop_enterprise", r"desktop|enterprise"),
    ("web_unspecified", r"\bweb\b"),
    ("data_engineer", r"engineer, data|data engineer|business intelligence"),
    ("data_science", r"data scien|machine learning|statistics|ai/ml|developer, ai|analyst"),
    ("devops_cloud", r"devops|site reliability|cloud infrastructure"),
    ("sysadmin_dba", r"system.? admin|database admin"),
    ("qa_test", r"\bqa\b|quality assurance"),
    ("management", r"manager|executive|c-suite"),
    ("student", r"student"),
    ("research_education", r"academic|research|scientist|educator"),
    ("designer", r"design"),
]
ROLE_RULES = [(name, re.compile(rule, re.I)) for name, rule in ROLE_RULES]
TECH_ALIASES = {
    "lang": {"HTML": "HTML/CSS", "CSS": "HTML/CSS", "HTML5": "HTML/CSS", "Bash/Shell (all shells)": "Bash/Shell",
             "Bash/Shell/PowerShell": "Bash/Shell", "PowerShell": "Bash/Shell", "Matlab": "MATLAB", "LISP": "Lisp",
             "Common Lisp": "Lisp", "VBA": "Visual Basic", "VB.NET": "Visual Basic", "Visual Basic (.Net)": "Visual Basic",
             "Visual Basic 6": "Visual Basic", "Delphi/Object Pascal": "Delphi", "COBOL": "Cobol", "Ocaml": "OCaml",
             "C++11": "C++", "Node.js": None},
    "db": {"SQL Server": "Microsoft SQL Server", "Dynamodb": "DynamoDB", "Amazon DynamoDB": "DynamoDB",
           "Firebase Realtime Database": "Firebase", "Cloud Firestore": "Firebase", "Neo4J": "Neo4j", "IBM Db2": "IBM DB2",
           "Google BigQuery": "BigQuery", "Couch DB": "CouchDB", "Microsoft Azure (Tables, CosmosDB, SQL, etc)": "Cosmos DB"},
    "platform": {"Amazon Web Services (AWS)": "AWS", "Azure": "Microsoft Azure", "Google Cloud Platform": "Google Cloud",
                 "Google Cloud Platform/App Engine": "Google Cloud", "Windows Desktop or Server": "Windows",
                 "Windows Desktop": "Windows", "Linux Desktop": "Linux", "Mac OS": "MacOS", "Digital Ocean": "DigitalOcean",
                 "Linode, now Akamai": "Linode", "Arduino / Raspberry Pi": "Arduino", "Wordpress": "WordPress"},
    "web": {"ReactJS": "React", "React.js": "React", "AngularJS": "Angular", "Angular.js": "Angular",
            "Angular/Angular.js": "Angular", "ASP.NET Core": "ASP.NET", "ASP.NET CORE": "ASP.NET", ".NET Core": "ASP.NET",
            "Spring Boot": "Spring", "Wordpress": "WordPress"},
}
OTHER_TOKEN = re.compile(r"^other\b", re.I)

In [7]:
# Functions that parse text answers into clean values.
def slug(text):
    text = text.replace("C++", "cpp").replace("C#", "csharp").replace("F#", "fsharp")
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def tidy(text):
    """Normalize a raw answer; U+FFFD marks apostrophes lost before the file was combined."""
    return re.sub(r"\s+", " ", text.replace("�", "'").replace("’", "'")).strip()


class Unmapped:
    """Collects answers no mapper recognized so they can be audited in the report."""
    NON_ANSWER = re.compile(r"prefer not|rather not|don.?t know|not sure|^other|something else|it.?s complicated|"
                            r"not part of a company|not working|^student|don.?t have a job|in your own words", re.I)

    def __init__(self):
        self.values = {}

    def apply(self, name, series, func):
        """Apply func to each distinct value; None means no usable value."""
        result = {}
        for value, count in series.value_counts().items():
            mapped = func(tidy(value))
            result[value] = mapped
            if mapped is None and not self.NON_ANSWER.search(tidy(value)):
                self.values.setdefault(name, {})[value[:80]] = int(count)
        return series.map(result)


def parse_years(text):
    low = text.lower()
    if low in EXCEL_DATE_YEARS:
        return EXCEL_DATE_YEARS[low]
    if low == "<2":
        return 1.0
    if low in ("less than 1 year", "less than a year"):
        return 0.5
    if low == "11+ years":
        return 13.0  # the open-ended top band of the early surveys; 2016 publishes 13 as its midpoint
    match = re.match(r"(\d+) or more years|more than (\d+) years", low)
    if match:
        return float(match.group(1) or match.group(2)) + 1
    match = re.match(r"(\d+)\s*(?:-|to)\s*(\d+) years", low)
    if match:
        return (float(match.group(1)) + float(match.group(2))) / 2
    try:
        return float(low)
    except ValueError:
        return None


def parse_age(text):
    low = re.sub(r"\s*-\s*", "-", text.lower())
    if low in AGE_BANDS:
        return float(AGE_BANDS[low])
    try:
        return float(low)
    except ValueError:
        return None


def parse_education(text):
    """Highest level mentioned. 0 none/primary ... 6 doctoral/professional."""
    levels = []
    for part in text.lower().split(";"):
        if re.search(r"doctoral|ph\.?d|professional degree", part):
            levels.append(6)
        elif "master" in part:
            levels.append(5)
        elif re.search(r"some college", part):
            levels.append(2)
        elif re.search(r"bachelor|b\.s\.|b\.a\.", part):
            levels.append(4)
        elif "associate" in part:
            levels.append(3)
        elif "secondary school" in part:
            levels.append(1)
        elif re.search(r"primary/elementary|never completed", part):
            levels.append(0)
    return float(max(levels)) if levels else None


def parse_employment(text):
    low = text.lower()
    if low in ("freelance / contractor", "self-employed"):
        return "contractor"
    if re.search(r"employed, (full|part)|employed (full|part)|^employed$", low):
        return "employed"
    if "contractor" in low:
        return "contractor"
    if "student" in low:
        return "student"
    if re.search(r"not employed|^unemployed", low):
        return "not_employed"
    if "retired" in low:
        return "retired"
    return None


def parse_gender(text):
    tokens = {t.strip().lower() for t in text.split(";")}
    if any(t.startswith("prefer not") for t in tokens):
        return None
    if tokens <= {"male", "man"}:
        return "man"
    if tokens <= {"female", "woman"}:
        return "woman"
    return "non_binary_or_other"


def parse_satisfaction(text):
    low = text.lower()
    if low in SATISFACTION_TEXT:
        return SATISFACTION_TEXT[low]
    try:
        return float(low) / 10  # 2017, 2024 and 2025 use a 0-10 scale
    except ValueError:
        return None


def parse_country(text):
    if COUNTRY_BUCKETS.match(text):
        return None
    return COUNTRY_ALIASES.get(text, text)


def parse_org_size(text):
    size = ORG_SIZES.get(text.lower())
    return float(np.log10(size)) if size else None

In [8]:
# Functions that remove bad rows and merge columns across years.
def load_raw(path, report):
    raw = pd.read_csv(path, dtype=str, low_memory=False)
    report["rows_raw"], report["columns_raw"] = len(raw), raw.shape[1]
    raw["survey_year"] = raw["survey_year"].astype(int)
    return raw


def remove_embedded_headers(raw, report):
    """The first row of each 2011-2015 survey is a sub-header. Keep 2015's labels to name its columns."""
    labels = raw[raw["survey_year"] == 2015].iloc[0].dropna().astype(str)
    if labels["Unnamed: 0"] != "Country":
        raise ValueError("2015 sub-header row not found where expected")
    drop = [raw.index[raw["survey_year"] == year][0] for year in range(2011, 2016)]
    for index, year in zip(drop[:4], range(2011, 2015)):
        country_col = "What Country do you live in?" if year == 2014 else "What Country or Region do you live in?"
        if raw.at[index, country_col] != "Response":
            raise ValueError(f"{year} sub-header row not found where expected")
    report["embedded_header_rows_removed"] = len(drop)
    return raw.drop(index=drop).reset_index(drop=True), labels


def remove_duplicates(raw, report):
    """Exact duplicates within a year, ignoring IDs. Sparse rows can match by chance, so they are left for MIN_ANSWERED."""
    drop = []
    for year, group in raw.groupby("survey_year"):
        ids = ID_COLUMNS + (["Unnamed: 0"] if year == 2016 else [])
        answers = group.drop(columns=ids + ["survey_year"]).dropna(axis=1, how="all")
        hashes = pd.util.hash_pandas_object(answers.fillna(""), index=False)
        duplicated = hashes.duplicated() & (answers.notna().sum(axis=1) >= 10)
        drop.extend(group.index[duplicated])
    report["duplicate_rows_removed"] = len(drop)
    return raw.drop(index=drop).reset_index(drop=True)


def add_2015_columns(raw, labels):
    """Name the 2015 columns that are only identified by the sub-header row."""
    is_2015 = raw["survey_year"] == 2015
    wanted = ["Country", "Age", "Gender", "Years IT / Programming Experience", "Occupation", "Employment Status",
              "Remote Status", "Job Satisfaction", "Compensation: midpoint"]
    new = {}
    for label in wanted:
        column = labels.index[labels == label][0]
        new[f"_2015_{label}"] = raw[column].where(is_2015)
    education_cols = labels.index[labels.str.startswith("Training & Education:")]
    new["_2015_education"] = raw.loc[is_2015, education_cols].apply(lambda row: ";".join(row.dropna()) or None, axis=1).reindex(raw.index)
    new["Salary_2017"] = raw["Salary"].where(raw["survey_year"] == 2017)  # 2018's Salary is in local currency
    return pd.concat([raw, pd.DataFrame(new)], axis=1)


def coalesce(raw, columns):
    result = raw[columns[0]]
    for column in columns[1:]:
        result = result.fillna(raw[column])
    return result


def harmonize(raw, report):
    unmapped = Unmapped()
    out = pd.DataFrame({"survey_year": raw["survey_year"]})
    get = {name: coalesce(raw, columns) for name, columns in SOURCES.items()}
    # In 2011-2014 "11" is the top band ("11+ years"); from 2019 on it means exactly 11 years.
    get["years_code"] = get["years_code"].mask((raw["survey_year"] <= 2014) & get["years_code"].eq("11"), "11+ years")

    country = unmapped.apply("country", get["country"], parse_country)
    top = country.value_counts().head(TOP_COUNTRIES).index
    out["country"] = country.where(country.isin(top) | country.isna(), "Other")
    out["age"] = unmapped.apply("age", get["age"], parse_age)
    out["gender"] = unmapped.apply("gender", get["gender"], parse_gender)
    out["employment"] = unmapped.apply("employment", get["employment"], parse_employment)
    out["education_level"] = get["education"].map(
        {value: parse_education(tidy(value)) for value in get["education"].dropna().unique()})
    out["org_size_log10"] = unmapped.apply("org_size", get["org_size"], parse_org_size)
    out["years_code"] = unmapped.apply("years_code", get["years_code"], parse_years)
    out["years_code_pro"] = unmapped.apply("years_code_pro", get["years_code_pro"], parse_years)
    out["remote_work"] = unmapped.apply("remote_work", get["remote_work"], lambda text: REMOTE.get(text.lower()))
    out["job_satisfaction"] = unmapped.apply("job_satisfaction", get["job_satisfaction"], parse_satisfaction)

    band = unmapped.apply("comp_band", get["comp_band"], lambda text: COMP_BANDS.get(text) or _to_float(text))
    exact = pd.to_numeric(get["comp_exact"], errors="coerce")
    out["comp_usd"] = exact.fillna(band).astype(float)
    for column in NUMERIC_ATTRS:
        out[column] = pd.to_numeric(out[column], errors="coerce").astype(float)
    report["unmapped_values"] = unmapped.values
    return out


def _to_float(text):
    try:
        return float(text)
    except ValueError:
        return None

In [9]:
# Load the raw CSV and remove embedded header rows and duplicates.
raw = load_raw(INPUT_CSV, report)
print(f"Loaded {report['rows_raw']:,} rows, {report['columns_raw']} columns")
raw, labels = remove_embedded_headers(raw, report)
raw = remove_duplicates(raw, report)
raw = add_2015_columns(raw, labels)
print(f"Removed {report['embedded_header_rows_removed']} header rows and {report['duplicate_rows_removed']:,} duplicates")

Loaded 772,672 rows, 946 columns
Removed 5 header rows and 21 duplicates


In [10]:
# Check that 2017 Salary is already in USD; if not, remove "Salary_2017" from SOURCES["comp_exact"].
check_2017 = raw.loc[raw["survey_year"] == 2017, ["Currency", "Salary"]].copy()
check_2017["Salary"] = pd.to_numeric(check_2017["Salary"], errors="coerce")
check_2017.groupby("Currency")["Salary"].agg(["count", "median"]).sort_values("count", ascending=False)

,count,median
Currency,,
U.S. dollars ($),4288,90000.000000
Euros (€),3182,40860.215054
British pounds sterling (£),1383,48750.000000
Indian rupees (?),795,7047.423286
Canadian dollars (C$),662,53030.303030
Polish zloty (zl),311,19753.086420
Australian dollars (A$),270,65909.090909
Brazilian reais (R$),236,15910.828025
Swedish kroner (SEK),194,49886.877828


In [11]:
# Functions that turn developer roles and technologies into 0/1 columns.
def assign_roles(raw):
    tokens = coalesce_tokens(raw, ROLE_SOURCES)
    role_of = {}
    for token in tokens.unique():
        text = tidy(token)
        role_of[token] = next((name for name, rule in ROLE_RULES if rule.search(text)), None)
    pairs = pd.DataFrame({"row": tokens.index, "name": tokens.map(role_of).values}).dropna().drop_duplicates()
    return to_multi_hot(pairs, [name for name, _ in ROLE_RULES], len(raw), "role_")


def coalesce_tokens(raw, columns):
    """All selected options as one Series of tokens, indexed by row."""
    parts = [raw[column].dropna().str.split(r"\s*;\s*").explode() for column in columns if column in raw]
    tokens = pd.concat(parts)
    return tokens[tokens.str.len() > 0]


def assign_tech(raw, labels, report):
    """Multi-hot over the top technologies per family, using harmonized names."""
    is_2015 = raw["survey_year"] == 2015
    current_2015 = [column for column in labels.index[labels.str.startswith("Current Lang & Tech:")]
                    if not labels[column].endswith("Write-In")]
    def selected_options(columns, mask):
        options = raw.loc[mask, columns].stack().dropna()
        options.index = options.index.get_level_values(0)
        return options

    mixed = [selected_options(current_2015, is_2015), raw["tech_do"].dropna().str.split(r"\s*;\s*").explode()]
    for year, (question_start, positions) in EARLY_LANGUAGE_QUESTIONS.items():
        question = next(column for column in raw.columns if column.startswith(question_start))
        columns = [question] + [f"Unnamed: {position}" for position in positions]
        mixed.append(selected_options(columns, raw["survey_year"] == year))
    mixed = pd.concat(mixed)
    mixed = mixed.map(lambda token: tidy(token))

    frames, vocabulary = [], {}
    for family, columns in TECH_SOURCES.items():
        tokens = coalesce_tokens(raw, columns).map(tidy)
        aliases = TECH_ALIASES[family]
        canonical = tokens.map(lambda token: aliases.get(token, token)).dropna()
        canonical = canonical[~canonical.str.match(OTHER_TOKEN)]
        counts = canonical.value_counts().rename_axis("token").reset_index(name="n").sort_values(["n", "token"], ascending=[False, True])
        vocab = counts["token"].head(TOP_TECH[family]).tolist()
        vocabulary[family] = vocab
        early = mixed.map(lambda token: aliases.get(token, token)).dropna()
        early = early[early.isin(vocab)]
        selected = pd.concat([canonical[canonical.isin(vocab)], early])
        pairs = pd.DataFrame({"row": selected.index, "name": selected.values}).drop_duplicates()
        frames.append(to_multi_hot(pairs, vocab, len(raw), f"{family}_"))
    report["technologies_kept"] = vocabulary
    return pd.concat(frames, axis=1)


def to_multi_hot(pairs, names, n_rows, prefix):
    positions = {name: position for position, name in enumerate(names)}
    matrix = np.zeros((n_rows, len(names)), dtype=np.uint8)
    matrix[pairs["row"].to_numpy(), pairs["name"].map(positions).to_numpy()] = 1
    return pd.DataFrame(matrix, columns=[prefix + slug(name) for name in names])

In [12]:
# Build the shared attributes plus the role and technology columns.
df = harmonize(raw, report)
roles = assign_roles(raw)
tech = assign_tech(raw, labels, report)
del raw
df.describe().T

,count,mean,std,min,25%,50%,75%,max
survey_year,772646.0,2020.026681,3.080231,2011.0,2018.000000,2020.000000,2023.000000,2.025000e+03
age,650450.0,31.405195,10.256025,1.0,22.000000,29.500000,39.500000,2.790000e+02
education_level,679640.0,3.649737,1.436306,0.0,2.000000,4.000000,5.000000,6.000000e+00
org_size_log10,530673.0,2.323053,1.194127,0.0,1.774517,2.476397,3.477049,4.176091e+00
years_code,733732.0,11.620747,9.390245,0.5,4.500000,9.000000,15.000000,1.000000e+02
years_code_pro,470268.0,8.871468,8.142257,0.5,3.000000,6.000000,12.000000,5.100000e+01
job_satisfaction,350757.0,0.687499,0.266360,0.0,0.500000,0.700000,0.900000,1.000000e+00
comp_usd,403960.0,102770.864677,426026.696231,0.0,27956.989247,58899.000000,100000.000000,7.435143e+07


In [13]:
# Function that nulls impossible values and caps compensation at per-year limits.
def handle_outliers(df, report):
    counts = {}

    def null_where(column, mask, label):
        counts[label] = int(mask.sum())
        df.loc[mask, column] = np.nan

    null_where("age", (df["age"] < AGE_RANGE[0]) | (df["age"] > AGE_RANGE[1]), "age_outside_range_nulled")
    null_where("years_code", df["years_code"] > MAX_YEARS, "years_code_too_large_nulled")
    null_where("years_code_pro", df["years_code_pro"] > MAX_YEARS, "years_code_pro_too_large_nulled")
    # Age bands are approximate, so allow 5 years of slack.
    null_where("years_code", df["years_code"] > df["age"] + 5, "years_code_exceeds_age_nulled")
    null_where("years_code_pro", df["years_code_pro"] > df["years_code"], "years_code_pro_exceeds_years_code_nulled")
    null_where("comp_usd", df["comp_usd"] <= 0, "comp_not_positive_nulled")

    # Compensation is heavy-tailed and full of currency and pay-period mistakes. Capping keeps
    # the respondent and the "very high/low" signal without letting bad values dominate.
    fences, capped = {}, 0
    for year, comp in df.loc[df["survey_year"] >= FIRST_EXACT_COMP_YEAR].groupby("survey_year")["comp_usd"]:
        logged = np.log10(comp.dropna())
        q1, q3 = logged.quantile([0.25, 0.75])
        low, high = 10 ** (q1 - IQR_K * (q3 - q1)), 10 ** (q3 + IQR_K * (q3 - q1))
        fences[int(year)] = [round(low), round(high)]
        capped += int(((comp < low) | (comp > high)).sum())
        df.loc[comp.index, "comp_usd"] = comp.clip(low, high)
    counts["comp_capped_at_fences"] = capped
    report["outliers"] = counts
    report["comp_fences_usd_by_year"] = fences
    return df

In [14]:
# Apply the outlier rules and show how many values changed.
df = handle_outliers(df, report)
print(json.dumps(report["outliers"], indent=2))
print("Compensation fences (USD):", report["comp_fences_usd_by_year"])

{
  "age_outside_range_nulled": 90,
  "years_code_too_large_nulled": 51,
  "years_code_pro_too_large_nulled": 0,
  "years_code_exceeds_age_nulled": 100,
  "years_code_pro_exceeds_years_code_nulled": 5189,
  "comp_not_positive_nulled": 1189,
  "comp_capped_at_fences": 21190
}
Compensation fences (USD): {2017: [5059, 419260], 2018: [3518, 680077], 2019: [3488, 748929], 2020: [3394, 701361], 2021: [3797, 711789], 2022: [5847, 735439], 2023: [9522, 560918], 2024: [5455, 647459], 2025: [6797, 677223]}


In [15]:
# Function that drops near-empty respondents, then imputes and flags missing values.
def handle_missing(df, roles, tech, report):
    core = NUMERIC_ATTRS + CATEGORICAL_ATTRS
    answered = df[core].notna().sum(axis=1) + (roles.sum(axis=1) > 0).astype(int) + (tech.sum(axis=1) > 0).astype(int)
    keep = answered >= MIN_ANSWERED
    report["near_empty_rows_removed"] = int((~keep).sum())
    df, roles, tech = (frame[keep].reset_index(drop=True) for frame in (df, roles, tech))
    report["missing_pct_before_imputation"] = df[core].isna().mean().mul(100).round(1).to_dict()
    report["missing_pct_by_year"] = df.groupby("survey_year")[core].agg(lambda s: round(s.isna().mean() * 100, 1)).to_dict("index")

    # Attributes are absent from whole survey years, so impute within the year and fall back to
    # the overall median. The flags keep real answers separable from imputed ones.
    for column in NUMERIC_ATTRS:
        df[f"{column}_missing"] = df[column].isna().astype(np.uint8)
        df[column] = df[column].fillna(df.groupby("survey_year")[column].transform("median")).fillna(df[column].median())
    for column in CATEGORICAL_ATTRS:
        df[column] = df[column].fillna("Unknown")
    df["roles_missing"] = (roles.sum(axis=1) == 0).astype(np.uint8)
    df["tech_missing"] = (tech.sum(axis=1) == 0).astype(np.uint8)
    return df, roles, tech

In [16]:
# Handle missing values and show the percent missing per attribute.
df, roles, tech = handle_missing(df, roles, tech, report)
print(f"Removed {report['near_empty_rows_removed']:,} near-empty rows; {len(df):,} remain")
pd.Series(report["missing_pct_before_imputation"], name="% missing before imputation")

Removed 12,587 near-empty rows; 760,059 remain


age                 14.9
years_code           3.6
years_code_pro      38.8
org_size_log10      30.2
education_level     10.6
job_satisfaction    53.9
comp_usd            47.0
country              3.1
employment           5.8
gender              37.8
remote_work         48.2
Name: % missing before imputation, dtype: float64

In [17]:
# Functions for one-hot encoding and PCA.
def encode_categoricals(df):
    dummies = pd.get_dummies(df[CATEGORICAL_ATTRS], prefix=CATEGORICAL_ATTRS, dtype=np.uint8)
    dummies.columns = [slug(column) for column in dummies.columns]
    return pd.concat([df.drop(columns=CATEGORICAL_ATTRS), dummies], axis=1)


def reduce_dimensions(multi_hot, output_dir, report):
    """Fit on respondents who answered, so "skipped the question" does not become the main component.

    Respondents who skipped get the mean, i.e. 0 on every component.
    """
    answered = multi_hot.sum(axis=1).to_numpy() > 0
    mean = multi_hot[answered].mean()
    std = multi_hot[answered].std().replace(0, 1)
    scaled = ((multi_hot - mean) / std).astype(np.float32)
    scaled.loc[~answered, :] = 0
    pca = PCA(svd_solver="covariance_eigh")
    pca.fit(scaled[answered])
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    n_components = int(np.searchsorted(cumulative, PCA_VARIANCE) + 1)
    components = scaled.to_numpy() @ pca.components_[:n_components].T
    names = [f"pc{i + 1}" for i in range(n_components)]
    pd.DataFrame(pca.components_[:n_components], index=names, columns=multi_hot.columns).round(4).to_csv(output_dir / "pca_loadings.csv")
    report["pca"] = {
        "input_columns": multi_hot.shape[1],
        "components_kept": n_components,
        "variance_explained": round(float(cumulative[n_components - 1]), 4),
        "first_10_components_variance": [round(float(v), 4) for v in pca.explained_variance_ratio_[:10]],
    }
    return pd.DataFrame(components, columns=names).round(DECIMALS)

In [18]:
# Reduce the role and technology columns with PCA.
components = reduce_dimensions(pd.concat([roles, tech], axis=1), OUTPUT_DIR, report)
print(f"PCA: {report['pca']['input_columns']} columns -> {report['pca']['components_kept']} components "
      f"({report['pca']['variance_explained']:.1%} of variance)")

PCA: 92 columns -> 74 components (90.5% of variance)


In [19]:
# Combine all features and save the preprocessed CSV and report.
final = pd.concat([encode_categoricals(df), components], axis=1)
for column in NUMERIC_ATTRS:
    final[column] = final[column].round(DECIMALS)
if final.isna().any().any():
    raise ValueError(f"Missing values remain: {final.columns[final.isna().any()].tolist()}")

report["rows_final"], report["columns_final"] = final.shape
report["rows_final_by_year"] = {int(k): int(v) for k, v in final["survey_year"].value_counts().sort_index().items()}
final.to_csv(OUTPUT_DIR / "survey_preprocessed.csv", index=False)
(OUTPUT_DIR / "preprocessing_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {final.shape[0]:,} rows x {final.shape[1]} columns in {time.time() - start:.0f}s")
final.head()

Wrote 760,059 rows x 132 columns in 107s


,survey_year,age,education_level,org_size_log10,years_code,years_code_pro,job_satisfaction,comp_usd,age_missing,years_code_missing,...,pc65,pc66,pc67,pc68,pc69,pc70,pc71,pc72,pc73,pc74
0,2011,17.0,4.0,1.114,1.0,6.0,0.00,70000.0,0,0,...,-0.113,-0.086,-0.520,-0.272,0.686,-0.528,-0.479,0.281,-1.019,0.188
1,2011,27.0,4.0,1.792,3.5,6.0,1.00,70000.0,0,0,...,0.647,0.491,-0.198,0.100,-0.566,-0.665,0.486,0.090,-0.139,0.343
2,2011,27.0,4.0,2.740,8.0,6.0,0.75,70000.0,0,0,...,0.690,0.341,0.258,-0.043,-0.469,-0.697,0.066,0.050,0.127,0.749
3,2011,17.0,4.0,1.792,3.5,6.0,0.75,70000.0,0,0,...,-0.153,-0.022,-0.090,0.175,-0.038,-0.228,0.028,0.059,0.142,0.090
4,2011,37.0,4.0,1.114,13.0,6.0,0.50,90000.0,0,0,...,-0.228,-0.157,0.311,-0.199,0.017,-0.076,0.447,-0.157,0.276,0.405


In [20]:
# List answers the mappings did not recognize.
for attribute, values in report["unmapped_values"].items():
    print(attribute, dict(sorted(values.items(), key=lambda item: -item[1])[:10]))

country {'South America': 498, 'Middle East': 395, 'Africa': 250, 'North America (Other)': 57, 'Central America': 44}
job_satisfaction {"I'm not happy in my job": 1030, 'I wish I had a job!': 897}
comp_band {'Unemployed': 1996}
